# 03-6 예외 처리 실습

오류 메시지와 traceback을 읽고, 예외를 알맞은 경계에서 처리하며, 입력 검증과 배치 처리 정책을 구현합니다. 모든 예외 실습은 `try`/`except`로 감싸 두었으므로 위에서부터 순서대로 전체 실행할 수 있습니다.

## 1. 오류 유형과 예외 객체 확인

In [ ]:
examples = [
    lambda: 10 / 0,
    lambda: int("three"),
    lambda: {"user": "min"}["role"],
]

observed_types = []
for operation in examples:
    try:
        operation()
    except Exception as exc:
        observed_types.append(type(exc).__name__)
        print(f"{type(exc).__name__}: {exc}")

assert observed_types == ["ZeroDivisionError", "ValueError", "KeyError"]

traceback은 마지막 줄의 **예외 유형과 메시지**를 먼저 읽고, 그 위의 호출 경로를 아래에서 위로 따라가면 원인을 찾기 쉽습니다.

## 2. 예외 전파와 처리 경계

In [ ]:
def divide(total, count):
    return total / count

def average(total, count):
    return divide(total, count)

try:
    average(100, 0)
except ZeroDivisionError as exc:
    propagated = type(exc).__name__
    print("프로그램 경계에서 처리:", propagated)

assert propagated == "ZeroDivisionError"

## 3. 여러 예외와 처리 순서

In [ ]:
def read_score(scores, key):
    try:
        return int(scores[key])
    except KeyError:
        return "키 없음"
    except (TypeError, ValueError):
        return "점수 형식 오류"

assert read_score({"kim": "90"}, "kim") == 90
assert read_score({}, "kim") == "키 없음"
assert read_score({"kim": "A"}, "kim") == "점수 형식 오류"
print("구체적인 예외부터 순서대로 처리했습니다.")

## 4. `else`와 `finally` 실행 흐름

In [ ]:
def convert_with_log(text):
    log = ["try"]
    try:
        number = int(text)
    except ValueError:
        log.append("except")
        result = None
    else:
        log.append("else")
        result = number * 2
    finally:
        log.append("finally")
    return result, log

assert convert_with_log("21") == (42, ["try", "else", "finally"])
assert convert_with_log("x") == (None, ["try", "except", "finally"])
print(convert_with_log("21"))
print(convert_with_log("x"))

In [ ]:
def return_and_cleanup(log):
    try:
        log.append("결과 계산")
        return "완료"
    finally:
        log.append("정리")

cleanup_log = []
assert return_and_cleanup(cleanup_log) == "완료"
assert cleanup_log == ["결과 계산", "정리"]
print(cleanup_log)

## 5. `raise`로 함수 계약 지키기

In [ ]:
def validate_port(port):
    if type(port) is not int:
        raise TypeError("port는 정수여야 합니다")
    if not 1 <= port <= 65535:
        raise ValueError("port는 1~65535 범위여야 합니다")
    return port

def expect_exception(exception_type, function, *args):
    try:
        function(*args)
    except exception_type as exc:
        return exc
    except Exception as exc:
        raise AssertionError(f"{exception_type.__name__} 대신 {type(exc).__name__} 발생") from exc
    raise AssertionError(f"{exception_type.__name__}이 발생하지 않음")

assert validate_port(443) == 443
assert str(expect_exception(TypeError, validate_port, "443")) == "port는 정수여야 합니다"
assert str(expect_exception(ValueError, validate_port, 70000)) == "port는 1~65535 범위여야 합니다"
print("정상·타입 오류·범위 오류를 모두 검증했습니다.")

## 6. 예외 연결과 원인 보존

In [ ]:
def parse_port(value):
    if type(value) not in {int, str}:
        raise TypeError("port는 정수 또는 문자열이어야 합니다")
    try:
        port = int(value)
    except ValueError as exc:
        raise ValueError("port를 정수로 변환할 수 없습니다") from exc
    if not 1 <= port <= 65535:
        raise ValueError("port는 1~65535 범위여야 합니다")
    return port

assert parse_port(443) == 443
assert parse_port("8080") == 8080
chained = expect_exception(ValueError, parse_port, "https")
assert isinstance(chained.__cause__, ValueError)
print("새 메시지:", chained)
print("원래 원인:", chained.__cause__)

In [ ]:
def log_and_reraise(log):
    try:
        int("not-a-number")
    except ValueError:
        log.append("변환 실패 기록")
        raise

reraised_log = []
reraised = expect_exception(ValueError, log_and_reraise, reraised_log)
assert reraised_log == ["변환 실패 기록"]
print(type(reraised).__name__, reraised_log)

## 7. LBYL과 EAFP

In [ ]:
record = {"user": "min", "role": "analyst"}
role_lbyl = record["role"] if "role" in record else "guest"
try:
    role_eafp = record["role"]
except KeyError:
    role_eafp = "guest"
assert role_lbyl == role_eafp == "analyst"
print("두 방식 모두 결과는 같지만, 상황에 맞게 가독성을 비교해야 합니다.")

## 8. 배치 처리 정책: best-effort

In [ ]:
def convert_many(values):
    converted = []
    errors = []
    for index, value in enumerate(values, start=1):
        try:
            converted.append(int(value))
        except (TypeError, ValueError) as exc:
            errors.append({"index": index, "value": value, "error_type": type(exc).__name__})
    return converted, errors

converted, conversion_errors = convert_many(["10", "x", 30, None])
assert converted == [10, 30]
assert [item["index"] for item in conversion_errors] == [2, 4]
print(converted)
print(conversion_errors)

## 9. `assert`와 실행 중 검증의 차이

In [ ]:
def register_age(age):
    if type(age) is not int:
        raise TypeError("age는 정수여야 합니다")
    if age < 0:
        raise ValueError("age는 0 이상이어야 합니다")
    return {"age": age}

assert register_age(20) == {"age": 20}  # 개발자의 기대를 검사
expect_exception(ValueError, register_age, -1)  # 외부 입력은 예외로 검증
print("외부 입력 검증을 assert에 맡기지 않았습니다.")

## 10. 미니 실습: 이벤트 행 파싱과 오류 보고

입력 형식은 `ACTION IP PORT`입니다. 정상 행은 구조화하고, 잘못된 행은 위치와 안전한 오류 정보만 기록합니다.

In [ ]:
ALLOWED_ACTIONS = {"ALLOW", "DENY"}

def normalize_action(value):
    if type(value) is not str:
        raise TypeError("action은 문자열이어야 합니다")
    action = value.upper()
    if action not in ALLOWED_ACTIONS:
        raise ValueError("action은 ALLOW 또는 DENY여야 합니다")
    return action

def parse_event_line(line):
    if type(line) is not str:
        raise TypeError("행은 문자열이어야 합니다")
    parts = line.split()
    if len(parts) != 3:
        raise ValueError("행은 ACTION IP PORT 세 항목이어야 합니다")
    action_text, ip, port_text = parts
    return {"action": normalize_action(action_text), "ip": ip, "port": parse_port(port_text)}

def parse_event_lines(lines):
    events = []
    errors = []
    for line_number, line in enumerate(lines, start=1):
        try:
            events.append(parse_event_line(line))
        except (TypeError, ValueError) as exc:
            errors.append({"line": line_number, "error_type": type(exc).__name__, "message": str(exc)})
    return events, errors

In [ ]:
sample_lines = [
    "ALLOW 10.0.0.1 443",
    "deny 10.0.0.2 22",
    "BLOCK 10.0.0.3 80",
    "ALLOW 10.0.0.4 https",
    "ALLOW 10.0.0.5 70000",
    "ALLOW 10.0.0.6",
]
events, event_errors = parse_event_lines(sample_lines)
assert events == [
    {"action": "ALLOW", "ip": "10.0.0.1", "port": 443},
    {"action": "DENY", "ip": "10.0.0.2", "port": 22},
]
assert [item["line"] for item in event_errors] == [3, 4, 5, 6]
assert sample_lines[0] == "ALLOW 10.0.0.1 443"
print("정상 이벤트:", events)
print("오류 보고:", event_errors)

## 11. 확장 실습: 오류 한도에서 중단

In [ ]:
def parse_event_lines_limited(lines, max_errors):
    if type(max_errors) is not int:
        raise TypeError("max_errors는 정수여야 합니다")
    if max_errors < 1:
        raise ValueError("max_errors는 1 이상이어야 합니다")
    events = []
    errors = []
    processed = 0
    for line_number, line in enumerate(lines, start=1):
        processed = line_number
        try:
            events.append(parse_event_line(line))
        except (TypeError, ValueError) as exc:
            errors.append({"line": line_number, "error_type": type(exc).__name__, "message": str(exc)})
            if len(errors) >= max_errors:
                break
    return {
        "events": events, "errors": errors, "processed": processed,
        "skipped": len(lines) - processed, "stopped_early": processed < len(lines),
    }

limited = parse_event_lines_limited(sample_lines, max_errors=2)
assert [item["line"] for item in limited["errors"]] == [3, 4]
assert limited["processed"] == 4
assert limited["skipped"] == 2
assert limited["stopped_early"] is True
print(limited)

## 12. 최종 점검

1. traceback의 마지막 줄부터 읽는 이유는 무엇인가요?
2. `except Exception`을 가장 앞에 두면 왜 문제가 되나요?
3. `else`와 `finally`는 각각 언제 실행되나요?
4. `raise ... from exc`가 보존하는 정보는 무엇인가요?
5. fail-fast와 best-effort 중 어떤 정책을 택할지 무엇을 기준으로 판단하나요?